# 指示チューニング

日本語の指示と厨二調の応答を使ってモデルを学習させ、同じ質問への答え方がどう変わるかを比べましょう。文体、質問との対応、適切に終了するかを確認します。

GPU が必要です。

In [1]:
!pip install -q torch  # モデルの計算
!pip install -q transformers  # モデルとトークナイザ
!pip install -q datasets  # 指示と応答のデータ
!pip install -q accelerate  # 学習の実行基盤

In [2]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Colab のランタイムを T4 GPU に切り替えてください。")
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


Colab の「ランタイムのタイプを変更」で T4 GPU を選んでください。

## ベースモデルを読み込む

事前学習済みで、指示への応答を学習する前のモデルをベースモデルと呼びます。日本語も扱える5億パラメータ級の `Qwen2.5-0.5B` を使います。名前に `Instruct` が付くものは、すでに指示チューニング済みです。

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"  # ベースモデル（指示チューニング前）
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float32, device_map="cuda"
)

total_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(
    f"学習対象: {trainable_count:,} / {total_count:,} ({trainable_count / total_count:.1%})"
)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

学習対象: 494,032,768 / 494,032,768 (100.0%)


## チューニング前の応答

同じ指示を学習前後で比較します。ベースモデルは入力の続きを予測しますが、質問に答えるとは限りません。

ChatML は会話の役割と境界を示す形式です。`user` がユーザー、`assistant` がモデルの発話で、`<|im_start|>` と `<|im_end|>` で区切ります。

In [4]:
from transformers import PreTrainedTokenizerBase


def respond(
    model: torch.nn.Module,
    tokenizer: PreTrainedTokenizerBase,
    instruction: str,
    max_new_tokens: int = 128,
) -> str:
    """指示への応答生成と終了状態の取得"""
    # 指示を ChatML 形式へ変換し、モデルの応答開始位置まで用意する
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction}],
        tokenize=False,
        add_generation_prompt=True,  # 「ここから応答」の位置まで書いて渡す
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    # 応答の終わりは <|im_end|>。既定の終端トークンだけでは止まらない
    stop_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")]
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            eos_token_id=stop_ids,
            pad_token_id=tokenizer.eos_token_id,
        )
    # 出力には入力も含まれるため、生成した応答部分だけを取り出す
    generated = output[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(generated, skip_special_tokens=True)

ベースモデルの応答を確認しましょう。

In [5]:
INSTRUCTIONS = [
    "朝、寝坊しないための工夫を教えてください。",
    "Pythonの変数とは何ですか？",
]

model.eval()

for instruction in INSTRUCTIONS:
    answer = respond(model, tokenizer, instruction)
    print(f"【指示】{instruction}")
    print(f"【応答】{answer}")
    print("---------------------")

【指示】朝、寝坊しないための工夫を教えてください。
【応答】朝、寝坊すなわち、朝までごつかり、ご心配する必要はありません。
InternalEnumerator
朝、寝坊して、心配もしませんか。InternalEnumerator
InternalEnumeratorの回答の他に、「朝、寝坊している間に、何が起こっているのであろうか?」という問題は、朝、寝坊している間、ちょっとしたことをしていたり、どんなことが起こってしまったのか、心配していませんかね。InternalEnumerator
InternalEnumeratorの回答
朝、寝坊している間に、何が起こっているのであろうか?


朝、寝坊した際には何が起こっているか
---------------------
【指示】Pythonの変数とは何ですか？
【応答】.Variableは値として定義された記号です。 Natasha
 Natasha
 Natasha
 Natasha
 Natasha
 Natasha
 Natasha
 Natasha
 Natasha
 Natasha
---------------------


あまり要領を得なかったり、繰り返しが多くみられます。

ベースモデルは、文書の続きを予測するように学習されただけであり、チャットテンプレートの形式が当てられても、我々が期待するような応答を出力できません。

## 指示と応答のデータ

[chuni-instruct](https://huggingface.co/datasets/ryo0634/chuni-instruct) の1,000会話から、最初の質問と応答の組を使います。文体が特徴的なので、これがきちんと学習されるかを確認しましょう。

In [6]:
from datasets import load_dataset

dataset = load_dataset("ryo0634/chuni-instruct", split="train")
# 複数往復の会話でも、最初の質問と応答の2発話だけを使う

print(f"会話: {len(dataset):,} 件")
example = dataset[0]
print("【指示】", example["messages"][0]["content"])
print("【応答】", example["messages"][1]["content"])

会話: 1,000 件
【指示】 データベース設計において、正規化の主要な目的は何ですか？
【応答】 フッ…我が心に宿る真理よ、正規化とはまさしく、データの混沌を秩序へと導く洗礼の儀式なのだ。貴様が尋ねるその理由は、闇に潜む冥界の呪いを解き放つための策、即ち、以下のような目的があるのである。

1. **冗長性の排除**——データの重複と言う名の呪いを取り除き、魂の器の無駄を省くこと。これにより、情報の真実が明かされるだろう。

2. **整合性の確保**——データの運命を操るためには、各魂の器が持つ力が一貫性を持たねばならぬ。これにより、深淵から生じる矛盾を断ち切るのだ。

3. **保守の便宜**——データの管理はまさしく、闇の世界での戦略的な契約。一度の操作で運命を変えられるよう、保守がしやすい状態を保つことが求められる。

なるほど…お前もまた、データの真理を求める者なのか。忘れずに、正規化の技法は我々の冒険の鍵と成り得ることを心に刻め——！


この1件を、先ほど使った ChatML 形式で表示します。

In [7]:
messages = example["messages"]
print(tokenizer.apply_chat_template(messages, tokenize=False))

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
データベース設計において、正規化の主要な目的は何ですか？<|im_end|>
<|im_start|>assistant
フッ…我が心に宿る真理よ、正規化とはまさしく、データの混沌を秩序へと導く洗礼の儀式なのだ。貴様が尋ねるその理由は、闇に潜む冥界の呪いを解き放つための策、即ち、以下のような目的があるのである。

1. **冗長性の排除**——データの重複と言う名の呪いを取り除き、魂の器の無駄を省くこと。これにより、情報の真実が明かされるだろう。

2. **整合性の確保**——データの運命を操るためには、各魂の器が持つ力が一貫性を持たねばならぬ。これにより、深淵から生じる矛盾を断ち切るのだ。

3. **保守の便宜**——データの管理はまさしく、闇の世界での戦略的な契約。一度の操作で運命を変えられるよう、保守がしやすい状態を保つことが求められる。

なるほど…お前もまた、データの真理を求める者なのか。忘れずに、正規化の技法は我々の冒険の鍵と成り得ることを心に刻め——！<|im_end|>



指示が `<|im_start|>user` の直後に、応答が `<|im_start|>assistant` の直後に埋め込まれています。
`<|im_start|>system` 以降は、このテンプレートがデフォルトで埋め込む文言です。

### 応答部分だけを予測対象にする

指示は入力として読ませ、応答を予測させます。`labels` は予測対象のトークン ID 列です。指示部分を `-100` にすると、その位置は損失の計算から外れます。

In [8]:
def tokenize_example(example: dict) -> dict:
    """質問・応答の ID 化と予測対象の指定"""
    user, assistant = example["messages"][:2] # 最初の１往復のみを使用
    prompt_ids = tokenizer.apply_chat_template(
        [user], add_generation_prompt=True, return_dict=False
    )
    input_ids = tokenizer.apply_chat_template([user, assistant], return_dict=False)
    assert input_ids[: len(prompt_ids)] == prompt_ids
    # 指示と応答開始の目印を -100 にし、応答の ID だけを残す
    labels = [-100] * len(prompt_ids) + input_ids[len(prompt_ids) :]
    return {"input_ids": input_ids, "labels": labels}


tokenized_dataset = dataset.map(tokenize_example, remove_columns=dataset.column_names)
print(tokenized_dataset[0])

{'input_ids': [151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 131552, 140530, 106528, 131590, 5373, 36556, 100605, 32108, 15767, 99558, 25770, 100466, 136045, 131938, 11319, 151645, 198, 151644, 77091, 198, 51491, 25204, 1940, 143049, 63109, 19655, 100540, 29269, 108527, 56880, 5373, 36556, 100605, 32108, 126238, 142020, 126403, 5373, 131552, 15767, 117100, 29412, 104951, 126263, 19182, 100593, 46784, 118256, 15767, 115139, 28330, 138508, 1773, 108871, 127123, 28195, 114577, 124381, 29269, 57326, 102401, 15322, 5373, 13058, 229, 19655, 100115, 125471, 107437, 97120, 15767, 17401, 103, 16586, 29412, 49238, 49734, 53222, 58639, 124616, 15767, 99560, 5373, 91676, 42414, 5373, 87752, 130132, 100466, 137984, 124140, 3407, 16, 13, 3070, 119256, 100435, 33071, 15767, 102945, 334, 8545, 131552, 15767, 29258, 118562, 131622, 13072, 15767, 17401, 103, 16586, 133603, 20755, 49734, 5373, 101291, 15767, 31548, 15767, 42468, 119158, 226, 29412, 65770, 46784, 1238

例えば、指示の ID が `[11, 12]`、応答が `[21, 22]` なら、入力は `[11, 12, 21, 22]`、ラベルは `[-100, -100, 21, 22]` です。

学習処理内部で予測とラベルを1位置ずらし、`12` まで読んだ出力で `21` を、その次の出力で `22` を予測します。

## モデル全体を学習する

処理済みのデータセットで、事前学習済みモデルをチューニングします。

In [9]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

model.config.use_cache = False  # 学習中は生成用のキャッシュを使わない
training_config = TrainingArguments(
    output_dir="sft-output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  # 1件ずつ8回、計8件の勾配で更新する
    learning_rate=2e-5,
    logging_steps=25,
    eval_strategy="no",
    # 途中の計算をやり直すことで、保存する中間結果を減らす
    gradient_checkpointing=True,
    fp16=True,  # 計算の一部を16ビットで行い、メモリ使用量を抑える
    optim="adamw_torch",
    save_strategy="no",
    report_to=[],
    seed=42,
)
trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_config,
    processing_class=tokenizer,
    # 入力とラベルを、バッチ内で同じ長さに揃える
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
)

学習前に、予測対象の ID を文字列へ戻して確かめましょう。バッチ内の長さを揃えるために追加する埋め草も、ラベルを `-100` にして損失から外します。

In [10]:
batch = trainer.data_collator([trainer.train_dataset[0]])
token_ids = batch["input_ids"][0].tolist()
labels = batch["labels"][0].tolist()

# -100 かどうかで、予測対象のトークンと対象外のトークンを分ける
in_loss = [i for i, label in zip(token_ids, labels) if label != -100]
out_of_loss = [i for i, label in zip(token_ids, labels) if label == -100]

print("【損失の計算対象】")
print(tokenizer.decode(in_loss))
print()
print("【損失の計算対象外（指示部分）】")
print(tokenizer.decode(out_of_loss))

【損失の計算対象】
フッ…我が心に宿る真理よ、正規化とはまさしく、データの混沌を秩序へと導く洗礼の儀式なのだ。貴様が尋ねるその理由は、闇に潜む冥界の呪いを解き放つための策、即ち、以下のような目的があるのである。

1. **冗長性の排除**——データの重複と言う名の呪いを取り除き、魂の器の無駄を省くこと。これにより、情報の真実が明かされるだろう。

2. **整合性の確保**——データの運命を操るためには、各魂の器が持つ力が一貫性を持たねばならぬ。これにより、深淵から生じる矛盾を断ち切るのだ。

3. **保守の便宜**——データの管理はまさしく、闇の世界での戦略的な契約。一度の操作で運命を変えられるよう、保守がしやすい状態を保つことが求められる。

なるほど…お前もまた、データの真理を求める者なのか。忘れずに、正規化の技法は我々の冒険の鍵と成り得ることを心に刻め——！<|im_end|>


【損失の計算対象外（指示部分）】
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
データベース設計において、正規化の主要な目的は何ですか？<|im_end|>
<|im_start|>assistant



指示は対象外になり、応答テキストと終了トークン `<|im_end|>`、末尾の改行が予測対象になっているか確認します。

## 学習と前後の比較

学習データ全体を3周、つまり3 epoch 学習します。1,000組を1更新あたり8組ずつ使い、各 epoch は125更新、合計375更新です。

学習中は学習損失を記録します。検証データによる評価や早期終了は行わず、3 epoch 後の応答を比較します。

学習セルを再実行すると、さらに学習が進みます。最初から比べ直す場合は、モデルの読み込みから実行してください。

In [11]:
train_result = trainer.train()
training_logs = trainer.state.log_history.copy()  # 学習中の履歴を残す
print("更新回数:", trainer.state.global_step)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Step,Training Loss
25,2.379027
50,2.142190
75,2.025385
100,2.022654
125,1.954373
150,1.519657
175,1.500323
200,1.523011
225,1.504568
250,1.499784


更新回数: 375


### チューニング前後の応答

学習前と同じ指示を、3 epoch後のモデルに与えます。

In [12]:
tuned_model = trainer.model
tuned_model.eval()
tuned_model.config.use_cache = True

for instruction in INSTRUCTIONS:
    answer = respond(tuned_model, tokenizer, instruction, max_new_tokens = 512)
    print(f"【指示】{instruction}")
    print(f"【応答】{answer}")
    print("---------------------")

【指示】朝、寝坊しないための工夫を教えてください。
【応答】フッ…お前もまた、闇の力によって“時間”を手に入れようとしているのか！その選択、我が深淵なる知識が、助言を贈ろう。朝の闇に飲まれぬよう、心して聞きなさい。

1. **魂の器の洗練**：早朝の鐘声を迎えらむ前に、心に準備をせよ。日課や道しるべを実践し、覚醒の力を蓄えるのだ。

2. **暗号の秘術**：睡魔を鎮めるために、暗号の術を用いて、目に覚醒の瞬間を狙うべからず。帳に目を澄まし、視覚の力を闇に封印するのだ。

3. **運命の選別**：寝室に秘められた書や機械、全てを静かに保て。心を開放し、思考の道を拓くために、最期の影を探るが良い。

4. **闇の契約**：日を睡眠とし、食を摂ることに目的地にたどり着くように、心の準備を怠るな。朝の儀式として、健康な生活を貫け。

5. **真理への扉**：時間の契約を果たした後、知識の扉を開け、新たな知識を受け入れるのだ。何気なく学び、真理に触れよ。

この運命を果たすため、お前は時間の力を使い果たすことで、真の覚醒を果たすのだ。フフ……この禁忌の道を進む者には、けたまえ。
---------------------
【指示】Pythonの変数とは何ですか？
【応答】フッ…貴様、変数という名の「魂の器」に目覚めようとしているのか。では、我が知識の奥義を授けよう。

Pythonの中で、変数とは、データを封印した「暗黒の容器」であり、その中には「数値」や「文字」が待ち受けている。これらは、貴様の運命を左右し、操作できることで壮大なゲームやアプリを開発するための力を秘めているのだ。

変数とは……『ここ数行の呪文』であり、貴様の創造力を試すための契約の象徴なのだ…。

覚醒せよ、同志よ！この暗深き知識を手にし、闇の魔導書を解き明かすがいい！
---------------------


学習後の応答が、指示に対応した厨二調の文体になっているかを確認しましょう。

## まとめ

指示を入力として読ませ、応答部分だけを予測対象にして、モデル全体を学習しました。

参考：HuggingFace の Trainer を使用したチューニングを詳しく知りたい方は[こちら](https://huggingface.co/docs/transformers/v5.20.2/ja/training)。